# Etapa 3 · Punto 2 — Integración de preferencias de usuarios (CritiqueBrainz)

**Objetivo.** Incorporar al grafo de Neo4j una fuente con **preferencias / interacciones de usuarios** sobre canciones,
reconciliar sus entidades con las canciones del grafo y preparar el **corpus documental** del componente RAG.

**Fuente:** [CritiqueBrainz](https://critiquebrainz.readthedocs.io/api/endpoints.html#reviews) (ecosistema MetaBrainz):
reseñas y calificaciones de usuarios sobre entidades identificadas con MBIDs de MusicBrainz.

**Flujo del notebook**

1. Descarga de reseñas de CritiqueBrainz (`recording`, `release_group`, `artist`) — con caché local.
2. Resolución de las entidades reseñadas en MusicBrainz (título + artistas acreditados).
3. **Reconciliación** con las canciones del grafo (4 estrategias, de más a menos específica) y **reporte de cobertura**.
4. Extensión de la ontología + generación de instancias RDF (`reviews.ttl`).
5. Carga en Neo4j (mismo mecanismo del punto 1: RDF → filas → `UNWIND`/`MERGE`) y materialización de la
   preferencia `(:User)-[:INTERACTED_WITH]->(:Track)` a partir de las cadenas de propiedades OWL.
6. **Corpus RAG:** limpieza, segmentación en *chunks*, *embeddings* e índice vectorial en Neo4j.
7. Recuperación de fragmentos relevantes (global y restringida a canciones candidatas) + evaluación rápida.

> Requiere haber ejecutado `01_carga_grafo_neo4j.ipynb` (el grafo base debe estar en Neo4j) y tener `entrega_3/.env`.

## 0. Entorno

In [15]:
%pip -q install "neo4j>=5.26,<7" "rdflib>=7,<8" "pandas>=2" requests "sentence-transformers>=3,<6"


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [16]:
import os, re, json, gzip, time, random, unicodedata
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import requests
pd.set_option('display.max_colwidth', 100)

DATA_DIR = Path(os.environ.get('DATA_DIR', '../data'))
CB_DIR = DATA_DIR / 'critiquebrainz'          # caché de descargas (CritiqueBrainz + MusicBrainz)
CB_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVO_MSD_MBID = DATA_DIR / 'msd_recording_mbid.csv'      # MSD track_id -> MusicBrainz recording MBID
ARCHIVO_ONTO_EXT = DATA_DIR / 'ontology_reviews.ttl'        # extensión de la ontología (este punto)
ARCHIVO_RESENAS_RDF = DATA_DIR / 'reviews.ttl'              # instancias generadas en este notebook
ARCHIVO_CORPUS = DATA_DIR / 'corpus_rag.jsonl.gz'           # chunks del corpus RAG (sin embeddings)

# Las APIs de MetaBrainz piden un User-Agent que identifique a la aplicación
HEADERS = {'User-Agent': 'semantic-web-course-project/1.0 (educational use)'}

In [17]:
# Conexión a Neo4j (igual que en el punto 1: credenciales desde entrega_3/.env)
from neo4j import GraphDatabase
from neo4j.exceptions import DriverError, ServiceUnavailable, SessionExpired
import logging
logging.getLogger('neo4j').setLevel(logging.CRITICAL)  # los cortes se reportan con nuestros propios mensajes de reintento

def cargar_env(ruta):
    for linea in ruta.read_text(encoding='utf-8').splitlines():
        if '=' in linea and not linea.lstrip().startswith('#'):
            k, v = linea.split('=', 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")

ARCHIVO_ENV = next((p for p in [Path('../.env'), Path('.env'), DATA_DIR / '.env'] if p.exists()), None)
if ARCHIVO_ENV is None:
    raise FileNotFoundError('No se encontró entrega_3/.env. Cópialo desde .env.example y completa las credenciales.')
cargar_env(ARCHIVO_ENV)
NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD = (os.environ[k] for k in ('NEO4J_URI', 'NEO4J_USER', 'NEO4J_PASSWORD'))
NEO4J_DATABASE = os.environ.get('NEO4J_DATABASE') or None

def conectar(intentos=15, espera=5):
    for i in range(1, intentos + 1):
        d = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD),
                                 connection_timeout=15, max_transaction_retry_time=120,
                                 notifications_min_severity='OFF')  # sin avisos de deprecación en la salida
        try:
            d.verify_connectivity()
            return d
        except (ServiceUnavailable, SessionExpired) as e:
            d.close()
            if i == intentos:
                raise
            print(f'Aura no disponible ({type(e).__name__}); reintento {i}/{intentos - 1} en {espera}s...')
            time.sleep(espera)

def sesion():
    global driver
    try:
        return driver.session(database=NEO4J_DATABASE)
    except DriverError:
        driver = conectar()
        return driver.session(database=NEO4J_DATABASE)

def con_reintentos(fn, intentos=15, espera=5):
    """Ejecuta fn(session); si Aura corta la conexión, reconecta y reintenta (las cargas usan MERGE: son idempotentes)."""
    global driver
    for i in range(1, intentos + 1):
        try:
            with sesion() as session:
                return fn(session)
        except (ServiceUnavailable, SessionExpired) as e:
            if i == intentos:
                raise
            print(f'Conexión perdida ({type(e).__name__}); reconectando {i}/{intentos - 1} en {espera}s...')
            time.sleep(espera)
            try:
                driver.close()
            except Exception:
                pass
            driver = conectar()

def run_cypher(query, **parameters):
    return con_reintentos(lambda s: [dict(r) for r in s.run(query, **parameters)])

def cargar_en_lotes(nombre, query, rows, batch_size=1000):
    t = time.time()
    for i in range(0, len(rows), batch_size):
        lote = rows[i:i + batch_size]
        con_reintentos(lambda s: s.execute_write(lambda tx: tx.run(query, rows=lote).consume()))
    print(f'✓ {nombre:<40} {len(rows):>8,}  ({time.time() - t:.1f}s)')

driver = conectar()
print('Conexión establecida con', driver.get_server_info().agent, '| usuario', NEO4J_USER)
n_tracks = run_cypher('MATCH (t:Track) RETURN count(t) AS n')[0]['n']
assert n_tracks > 0, 'El grafo base no está cargado: ejecuta primero 01_carga_grafo_neo4j.ipynb'
print(f'Canciones en el grafo: {n_tracks:,}')

Conexión establecida con Neo4j/5.27-aura | usuario 218cfcc4
Canciones en el grafo: 100,000


## 1. Descarga de reseñas de CritiqueBrainz

Endpoint `GET /ws/1/review/?entity_type=<tipo>&limit=50&offset=<n>`. Se descargan los tres tipos de entidad que pueden
vincularse a canciones del grafo:

| `entity_type` | Qué reseña | Vínculo posible con el grafo |
|---|---|---|
| `recording` | una grabación (canción) | directo: `recording_mbid` ↔ `Track` |
| `release_group` | un álbum | `Track` que pertenece a ese álbum (`rbo:released`) |
| `artist` | un artista | `Artist` (MBID) → sus `Track` |

Las respuestas se guardan recortadas (solo los campos necesarios) en `data/critiquebrainz/reviews_<tipo>.jsonl.gz`;
si el archivo existe, se reutiliza (no se vuelve a llamar a la API).

In [18]:
from email.utils import parsedate_to_datetime

CB_API = 'https://critiquebrainz.org/ws/1/review/'

def get_json(url, params, intentos=6):
    for i in range(intentos):
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=60)
            if r.status_code == 200:
                return r.json()
            if r.status_code not in (429, 500, 502, 503, 504):
                r.raise_for_status()
        except requests.RequestException as e:
            print('  error de red:', e)
        time.sleep(3 * (i + 1))
    raise RuntimeError(f'No se pudo obtener {url} {params}')

def recortar(r):
    pub = r.get('published_on') or r.get('created')
    return {'id': r['id'], 'entity_id': r['entity_id'], 'entity_type': r['entity_type'],
            'text': (r.get('text') or '').strip() or None, 'rating': r.get('rating'),
            'language': r.get('language'), 'license': r.get('license_id'),
            'published_on': parsedate_to_datetime(pub).isoformat() if pub else None,
            'source': r.get('source'), 'source_url': r.get('source_url'),
            'votes_positive': r.get('votes_positive_count'), 'votes_negative': r.get('votes_negative_count'),
            'user_id': r['user']['id'], 'user_name': r['user'].get('display_name')}

def descargar_resenas(entity_type):
    cache = CB_DIR / f'reviews_{entity_type}.jsonl.gz'
    if cache.exists():
        with gzip.open(cache, 'rt', encoding='utf-8') as f:
            return [json.loads(l) for l in f]
    resenas, offset = [], 0
    while True:
        datos = get_json(CB_API, {'entity_type': entity_type, 'limit': 50, 'offset': offset})
        resenas += [recortar(r) for r in datos['reviews']]
        offset += 50
        if offset >= datos['count']:
            break
        time.sleep(0.2)
    with gzip.open(cache, 'wt', encoding='utf-8') as f:
        for r in resenas:
            f.write(json.dumps(r, ensure_ascii=False) + '\n')
    return resenas

def sin_duplicados(resenas):
    # La paginación por offset puede repetir reseñas si cambian durante la descarga: se deduplica por id.
    return list({r['id']: r for r in resenas}.values())

RESENAS = {t: sin_duplicados(descargar_resenas(t)) for t in ('recording', 'release_group', 'artist')}
resumen = pd.DataFrame([{
    'entity_type': t, 'reseñas': len(rs), 'con_texto': sum(1 for r in rs if r['text']),
    'con_calificación': sum(1 for r in rs if r['rating']), 'entidades_distintas': len({r['entity_id'] for r in rs}),
    'usuarios': len({r['user_id'] for r in rs})} for t, rs in RESENAS.items()])
display(resumen)

,entity_type,reseñas,con_texto,con_calificación,entidades_distintas,usuarios
0,recording,916,361,870,914,264
1,release_group,13051,11761,3670,12607,1628
2,artist,633,387,579,600,523


## 2. Resolución de entidades en MusicBrainz

CritiqueBrainz solo entrega el MBID de la entidad reseñada. Para poder comparar con el grafo se consulta MusicBrainz y se
obtienen el **título** y los **MBIDs de los artistas acreditados**. Para no hacer miles de llamadas (límite: 1 petición/s),
se usa la API de búsqueda con hasta 100 identificadores por consulta (`rgid:A OR rgid:B ...`).
Resultado en caché: `data/critiquebrainz/musicbrainz_<tipo>.json` (solo se consultan los IDs que falten).

In [19]:
MB_API = 'https://musicbrainz.org/ws/2/'

def resolver_mb(tipo, campo, clave, ids):
    cache = CB_DIR / f'musicbrainz_{tipo}.json'
    meta = json.loads(cache.read_text(encoding='utf-8')) if cache.exists() else {}
    consultados = set(meta.get('_consultados', meta.keys()))
    pendientes = sorted(set(ids) - consultados)
    for i in range(0, len(pendientes), 100):
        lote = pendientes[i:i + 100]
        datos = get_json(MB_API + tipo.replace('_', '-'),
                         {'query': ' OR '.join(f'{campo}:{x}' for x in lote), 'limit': 100, 'fmt': 'json'})
        for e in datos.get(clave, []):
            meta[e['id']] = {'title': e['title'], 'artists': [c['artist']['id'] for c in e.get('artist-credit', [])]}
        time.sleep(1.1)
    if pendientes:
        cache.write_text(json.dumps(meta, ensure_ascii=False), encoding='utf-8')
    return {k: v for k, v in meta.items() if not k.startswith('_')}

MB_META = {
    'release_group': resolver_mb('release_group', 'rgid', 'release-groups', {r['entity_id'] for r in RESENAS['release_group']}),
    'recording': resolver_mb('recording', 'rid', 'recordings', {r['entity_id'] for r in RESENAS['recording']}),
}
for t, meta in MB_META.items():
    total = len({r['entity_id'] for r in RESENAS[t]})
    print(f'{t:<14} resueltas {len(meta):>6,} de {total:>6,} entidades ({len(meta) / total:.1%}); '
          'las restantes fueron fusionadas o eliminadas en MusicBrainz')

release_group  resueltas 12,406 de 12,607 entidades (98.4%); las restantes fueron fusionadas o eliminadas en MusicBrainz
recording      resueltas    890 de    914 entidades (97.4%); las restantes fueron fusionadas o eliminadas en MusicBrainz


## 3. Reconciliación con las canciones del grafo

Las canciones del grafo vienen del *Million Song Dataset* (ID `TR...`), que **no** trae MBIDs de recording. Se usan cuatro
estrategias, de mayor a menor especificidad; cada reseña queda con la estrategia que la emparejó (`match_method`):

| # | Estrategia | Clave de cruce | Confiabilidad |
|---|---|---|---|
| 1 | `recording_mbid` | MBID de la reseña ∈ mapeo oficial MSD → MusicBrainz (`msd-mbid-2016-01`, usado en la Etapa 1 para AcousticBrainz) | Exacta (identificador) |
| 2 | `recording_artista_titulo` | MBID de artista acreditado + título normalizado de la canción | Alta: el artista se cruza por identificador |
| 3 | `album_artista_titulo` | MBID de artista acreditado + título normalizado del álbum (`rbo:released`) | Alta para el álbum; la reseña habla del álbum que **contiene** la canción |
| 4 | `artista_mbid` | MBID del artista reseñado = `Artist.id` | Exacta, pero la reseña es sobre el artista en general |

**Normalización de títulos:** Unicode NFKD sin tildes, minúsculas, sin contenido entre paréntesis/corchetes
(`(Remastered)`, `[Live]`…), sin sufijos tipo `- 2004 Remaster`, y solo caracteres alfanuméricos. Nunca se cruzan títulos
que quedan vacíos tras normalizar. Exigir **siempre** el MBID del artista evita falsos positivos por títulos genéricos.

In [20]:
def normalizar(s):
    if not isinstance(s, str) or not s:          # None / NaN (pandas 3 usa NaN para textos faltantes)
        return ''
    s = unicodedata.normalize('NFKD', str(s))
    s = ''.join(c for c in s if not unicodedata.combining(c)).casefold()
    s = re.sub(r'\s*[\(\[].*?[\)\]]', ' ', s)                                         # (Remastered) [Live]
    s = re.sub(r'\s+-\s+.*\b(remaster\w*|version|edition|live|mix|edit|mono|stereo)\b.*$', ' ', s)
    s = re.sub(r'^the\s+', '', s.strip())
    return re.sub(r'[\W_]+', ' ', s).strip()

for ejemplo in ['The Headlight Serenade (Deluxe Edition)', 'Wish You Were Here - 2011 Remaster', 'Café Tacvba', '東京']:
    print(f'{ejemplo!r:45} -> {normalizar(ejemplo)!r}')

'The Headlight Serenade (Deluxe Edition)'     -> 'headlight serenade'
'Wish You Were Here - 2011 Remaster'          -> 'wish you were here'
'Café Tacvba'                                 -> 'cafe tacvba'
'東京'                                          -> '東京'


In [21]:
# Canciones del grafo (fuente de verdad: Neo4j, cargado en el punto 1)
tracks = pd.DataFrame(run_cypher('''
MATCH (t:Track)-[:SUNG_BY]->(a:Artist)
RETURN t.id AS track_id, t.title AS title, t.released AS album, a.id AS artist_mbid, a.name AS artist'''))

idx_titulo, idx_album, idx_artista = defaultdict(set), defaultdict(set), defaultdict(set)
for t in tracks.itertuples():
    idx_artista[t.artist_mbid].add(t.track_id)
    if normalizar(t.title):
        idx_titulo[(t.artist_mbid, normalizar(t.title))].add(t.track_id)
    if normalizar(t.album):
        idx_album[(t.artist_mbid, normalizar(t.album))].add(t.track_id)

msd_mbid = pd.read_csv(ARCHIVO_MSD_MBID, dtype=str)
idx_recording = msd_mbid.groupby('recording_mbid')['track_id'].apply(set).to_dict()
print(f'Canciones: {len(tracks):,} | con recording MBID (mapeo MSD): {msd_mbid.track_id.nunique():,} '
      f'| claves artista+título: {len(idx_titulo):,} | claves artista+álbum: {len(idx_album):,}')

Canciones: 100,000 | con recording MBID (mapeo MSD): 24,910 | claves artista+título: 97,618 | claves artista+álbum: 67,773


In [22]:
def cruzar_por_titulo(meta, indice):
    # Une los tracks de todos los artistas acreditados cuyo título normalizado coincide.
    if not meta or not normalizar(meta['title']):
        return set()
    return set().union(*[indice.get((a, normalizar(meta['title'])), set()) for a in meta['artists']])

def reconciliar(r):
    # Devuelve (método, tracks, diagnóstico) para una reseña.
    tipo, mbid = r['entity_type'], r['entity_id']
    if tipo == 'recording':
        if mbid in idx_recording:
            return 'recording_mbid', idx_recording[mbid], 'ok'
        meta = MB_META['recording'].get(mbid)
        ts = cruzar_por_titulo(meta, idx_titulo)
        if ts:
            return 'recording_artista_titulo', ts, 'ok'
    elif tipo == 'release_group':
        meta = MB_META['release_group'].get(mbid)
        ts = cruzar_por_titulo(meta, idx_album)
        if ts:
            return 'album_artista_titulo', ts, 'ok'
    elif tipo == 'artist':
        if mbid in idx_artista:
            return 'artista_mbid', idx_artista[mbid], 'ok'
        return None, set(), 'artista no está en el grafo'
    # Diagnóstico de por qué no se emparejó (recording / release_group)
    meta = MB_META[tipo].get(mbid)
    if meta is None:
        return None, set(), 'entidad no resuelta en MusicBrainz'
    if not any(a in idx_artista for a in meta['artists']):
        return None, set(), 'artista no está en el grafo'
    return None, set(), 'artista en el grafo, título no coincide'

filas = []
for tipo, resenas in RESENAS.items():
    for r in resenas:
        metodo, ts, diag = reconciliar(r)
        filas.append({**r, 'match_method': metodo, 'track_ids': sorted(ts), 'diagnostico': diag,
                      'mb_title': (MB_META.get(tipo, {}).get(r['entity_id']) or {}).get('title')})
cruce = pd.DataFrame(filas)
cruce = cruce.astype(object).where(cruce.notna(), None)   # NaN -> None (evita literales "NaN" en el RDF)
emparejadas = cruce[cruce.match_method.notna()].copy()
print(f'Reseñas emparejadas: {len(emparejadas):,} de {len(cruce):,}')

Reseñas emparejadas: 2,006 de 14,600


### 3.1 Reporte de cobertura

In [23]:
ORDEN = ['recording_mbid', 'recording_artista_titulo', 'album_artista_titulo', 'artista_mbid']
con_texto = emparejadas[emparejadas.text.notna()]

def tracks_de(df):
    return set().union(*df.track_ids.map(set)) if len(df) else set()

cobertura = pd.DataFrame([{
    'estrategia': m,
    'reseñas': (sub := emparejadas[emparejadas.match_method == m]).shape[0],
    'reseñas_con_texto': sub.text.notna().sum(),
    'entidades': sub.entity_id.nunique(),
    'usuarios': sub.user_id.nunique(),
    'canciones_cubiertas': len(tracks_de(sub)),
    'canciones_con_texto': len(tracks_de(sub[sub.text.notna()])),
} for m in ORDEN])
total = {'estrategia': 'TOTAL (sin duplicar)', 'reseñas': len(emparejadas), 'reseñas_con_texto': len(con_texto),
         'entidades': emparejadas.entity_id.nunique(), 'usuarios': emparejadas.user_id.nunique(),
         'canciones_cubiertas': len(tracks_de(emparejadas)), 'canciones_con_texto': len(tracks_de(con_texto))}
cobertura = pd.concat([cobertura, pd.DataFrame([total])], ignore_index=True)
display(cobertura)

n_cub, n_txt = total['canciones_cubiertas'], total['canciones_con_texto']
artistas_cub = tracks[tracks.track_id.isin(tracks_de(con_texto))].artist_mbid.nunique()
print(f'Cobertura sobre el grafo: {n_cub:,} canciones con al menos una reseña/calificación ({n_cub / len(tracks):.2%}); '
      f'{n_txt:,} con al menos una reseña con texto ({n_txt / len(tracks):.2%}), de {artistas_cub:,} artistas.')

,estrategia,reseñas,reseñas_con_texto,entidades,usuarios,canciones_cubiertas,canciones_con_texto
0,recording_mbid,7,0,7,3,7,0
1,recording_artista_titulo,7,4,7,5,7,4
2,album_artista_titulo,1942,1845,1849,348,2966,2889
3,artista_mbid,50,18,48,40,313,139
4,TOTAL (sin duplicar),2006,1867,1911,387,3206,2999


Cobertura sobre el grafo: 3,206 canciones con al menos una reseña/calificación (3.21%); 2,999 con al menos una reseña con texto (3.00%), de 1,419 artistas.


In [24]:
print('Diagnóstico de las reseñas NO emparejadas, por tipo de entidad')
display(cruce[cruce.match_method.isna()].groupby(['entity_type', 'diagnostico']).size()
        .rename('reseñas').reset_index())

print('Idiomas de las reseñas emparejadas con texto')
display(con_texto.language.value_counts().head(8).rename('reseñas').to_frame().T)
print('Fuente original (CritiqueBrainz importó críticas de la BBC con licencia CC)')
display(con_texto.source.fillna('usuario CritiqueBrainz').value_counts().rename('reseñas').to_frame().T)

Diagnóstico de las reseñas NO emparejadas, por tipo de entidad


,entity_type,diagnostico,reseñas
0,artist,artista no está en el grafo,583
1,recording,"artista en el grafo, título no coincide",168
2,recording,artista no está en el grafo,710
3,recording,entidad no resuelta en MusicBrainz,24
4,release_group,"artista en el grafo, título no coincide",4457
5,release_group,artista no está en el grafo,6448
6,release_group,entidad no resuelta en MusicBrainz,204


Idiomas de las reseñas emparejadas con texto


language,en,fr,es,de,cs,pt
reseñas,1831,27,5,2,1,1


Fuente original (CritiqueBrainz importó críticas de la BBC con licencia CC)


source,BBC,usuario CritiqueBrainz
reseñas,1677,190


In [25]:
# Validación manual: muestra aleatoria de emparejamientos por título (estrategias 2 y 3)
info = tracks.set_index('track_id')
muestra = emparejadas[emparejadas.match_method.isin(['recording_artista_titulo', 'album_artista_titulo'])].sample(12, random_state=7)
display(pd.DataFrame([{
    'estrategia': r.match_method, 'título MusicBrainz': r.mb_title,
    'artista (grafo)': info.loc[r.track_ids[0], 'artist'],
    'álbum/título en el grafo': info.loc[r.track_ids[0], 'album' if r.match_method.startswith('album') else 'title'],
    'n_canciones': len(r.track_ids)} for r in muestra.itertuples()]))

,estrategia,título MusicBrainz,artista (grafo),álbum/título en el grafo,n_canciones
0,album_artista_titulo,Elastica,Elastica,Elastica,1
1,album_artista_titulo,Introducing the Hardline According to Terence Trent D’Arby,Terence Trent D'Arby,Introducing The Hardline According To Terence Trent D'Arby,1
2,album_artista_titulo,The Real Thing,Vanessa Williams,The Real Thing,3
3,album_artista_titulo,Intensive Care,Robbie Williams,Intensive Care,1
4,album_artista_titulo,Long Road Out of Eden,Eagles,Long Road Out Of Eden,3
5,album_artista_titulo,New Morning,Sabrina Malheiros,New Morning,1
6,album_artista_titulo,The Soft Machine,Soft Machine,The Soft Machine,1
7,album_artista_titulo,Time Flies When You're Having Fun,Smokey Robinson,Time Flies When You're Having Fun,2
8,album_artista_titulo,Love the Woman,Chanté Moore,Love The Woman,1
9,album_artista_titulo,Souvlaki,Slowdive,Souvlaki,1


## 4. Extensión de la ontología y generación de RDF

`data/ontology_reviews.ttl` extiende la ontología de la Etapa 2 con:

- **Clases:** `res:User`, `res:Review`, `res:Album`.
- **Propiedades:** `rbo:writtenBy` (Review → User, inversa `rbo:wrote`), `rbo:reviews` y sus subpropiedades
  `rbo:reviewsRecording` / `rbo:reviewsAlbum` / `rbo:reviewsArtist`, `rbo:appearsOn` (Track → Album, inversa `rbo:hasTrack`),
  `rbo:releasedBy` (Album → Artist), datos de la reseña (`reviewText`, `rating`, `language`, `license`, `publishedOn`,
  `reviewUrl`, `reviewSource`, `matchMethod`) y `rbo:recordingMbid` en `Track`.
- **Preferencia usuario → canción** `rbo:interactedWith`, definida con tres **cadenas de propiedades** OWL:
  `wrote ∘ reviewsRecording`, `wrote ∘ reviewsAlbum ∘ hasTrack` y `wrote ∘ reviewsArtist ∘ sing`.

Las instancias se generan con `rdflib` (mismo estilo que la Etapa 2) en `data/reviews.ttl`.

In [26]:
from rdflib import Graph, Namespace, RDF, RDFS, OWL, XSD, URIRef, BNode, Literal
from rdflib.collection import Collection

RES = Namespace('http://uniandes.edu.co/wsproject2026/vocab/lit#')
RBO = Namespace('http://uniandes.edu.co/wsproject2026/vocab/ontology#')
onto_ext = Graph().parse(ARCHIVO_ONTO_EXT, format='turtle')

g = Graph()
g.bind('res', RES); g.bind('rbo', RBO)
PROP_OBJETIVO = {'recording': RBO.reviewsRecording, 'release_group': RBO.reviewsAlbum, 'artist': RBO.reviewsArtist}

for r in emparejadas.itertuples():
    review = RES[f'Review_{r.id}']
    g.add((review, RDF.type, RES.Review))
    g.add((review, RBO.mbid, Literal(r.id)))
    g.add((review, RBO.matchMethod, Literal(r.match_method)))
    g.add((review, RBO.reviewUrl, Literal(f'https://critiquebrainz.org/review/{r.id}', datatype=XSD.anyURI)))
    if r.text:
        g.add((review, RBO.reviewText, Literal(r.text, lang=r.language or None)))
    if r.rating:
        g.add((review, RBO.rating, Literal(int(r.rating), datatype=XSD.integer)))
    for prop, valor in [(RBO.language, r.language), (RBO.license, r.license), (RBO.reviewSource, r.source)]:
        if valor:
            g.add((review, prop, Literal(valor)))
    if r.published_on:
        g.add((review, RBO.publishedOn, Literal(r.published_on, datatype=XSD.dateTime)))

    user = RES[f'User_{r.user_id}']
    g.add((user, RDF.type, RES.User))
    g.add((user, RBO.mbid, Literal(r.user_id)))
    if r.user_name:
        g.add((user, RBO.displayName, Literal(r.user_name)))
    g.add((review, RBO.writtenBy, user))

    if r.entity_type == 'recording':
        for tid in r.track_ids:
            g.add((review, RBO.reviewsRecording, RES[f'Track_{tid}']))
    elif r.entity_type == 'artist':
        g.add((review, RBO.reviewsArtist, RES[f'Artist_{r.entity_id}']))
    else:
        album = RES[f'Album_{r.entity_id}']
        meta = MB_META['release_group'][r.entity_id]
        g.add((album, RDF.type, RES.Album))
        g.add((album, RBO.mbid, Literal(r.entity_id)))
        g.add((album, RDFS.label, Literal(meta['title'])))
        g.add((review, RBO.reviewsAlbum, album))
        for a in meta['artists']:
            if a in idx_artista:
                g.add((album, RBO.releasedBy, RES[f'Artist_{a}']))
        for tid in r.track_ids:
            g.add((RES[f'Track_{tid}'], RBO.appearsOn, album))

# Clave de reconciliación: recording MBIDs de cada canción (mapeo MSD -> MusicBrainz)
for fila in msd_mbid.itertuples():
    g.add((RES[f'Track_{fila.track_id}'], RBO.recordingMbid, Literal(fila.recording_mbid)))

g.serialize(destination=ARCHIVO_RESENAS_RDF, format='turtle')
print(f'Triples generados: {len(g):,} -> {ARCHIVO_RESENAS_RDF}')
display(pd.DataFrame(Counter(local for local in (str(o).split('#')[-1] for o in g.objects(None, RDF.type))).most_common(),
                     columns=['clase', 'instancias']))

Triples generados: 73,233 -> ../data/reviews.ttl


,clase,instancias
0,Review,2006
1,Album,1849
2,User,387


In [27]:
# Validación: todo lo usado en reviews.ttl está declarado en la ontología (base + extensión)
onto_base = Graph().parse(DATA_DIR / 'ontology.ttl', format='turtle')
declarado = set(onto_base.subjects()) | set(onto_ext.subjects())
usados = {p for p in g.predicates() if p not in (RDF.type, RDFS.label)} | set(g.objects(None, RDF.type))
no_declarados = sorted(str(x).split('#')[-1] for x in usados - declarado)
print('Términos no declarados en la ontología:', no_declarados or 'ninguno ✓')

# Validación de dominios/rangos de las propiedades de objeto de la extensión
violaciones = 0
tipo = lambda x: set(g.objects(x, RDF.type)) or ({RES.Track} if '#Track_' in str(x) else {RES.Artist} if '#Artist_' in str(x) else set())
for p in onto_ext.subjects(RDF.type, OWL.ObjectProperty):
    d, rg_ = onto_ext.value(p, RDFS.domain), onto_ext.value(p, RDFS.range)
    for s, o in g.subject_objects(p):
        violaciones += (d is not None and d not in tipo(s)) + (rg_ is not None and rg_ not in tipo(o))
print('Violaciones de dominio/rango:', violaciones)

Términos no declarados en la ontología: ninguno ✓
Violaciones de dominio/rango: 0


## 5. Carga en Neo4j

Mismo mecanismo del punto 1: se leen los triples de `reviews.ttl`, se convierten en filas y se importan con `UNWIND`/`MERGE`.

| RDF | Neo4j |
|---|---|
| `res:User` | `(:User {uri, id, name})` |
| `res:Review` | `(:Review {uri, id, text, rating, language, license, published_on, url, source, match_method})` |
| `res:Album` | `(:Album {uri, id, name})` |
| `rbo:writtenBy` | `(:Review)-[:WRITTEN_BY]->(:User)` |
| `rbo:reviewsRecording` / `reviewsAlbum` / `reviewsArtist` | `(:Review)-[:REVIEWS_RECORDING]->(:Track)`, `-[:REVIEWS_ALBUM]->(:Album)`, `-[:REVIEWS_ARTIST]->(:Artist)` |
| `rbo:appearsOn`, `rbo:releasedBy` | `(:Track)-[:APPEARS_ON]->(:Album)`, `(:Album)-[:RELEASED_BY]->(:Artist)` |
| `rbo:recordingMbid` | `Track.recordingMbids` (lista) |
| `rbo:interactedWith` (cadenas OWL) | `(:User)-[:INTERACTED_WITH {via, rating, review}]->(:Track)` **materializada** con Cypher |

In [28]:
local = lambda u: str(u).split('#')[-1]
tipo_relacion = lambda nombre: re.sub(r'(?<!^)(?=[A-Z])', '_', nombre).upper()
val = lambda s, p: (lambda v: None if v is None else v.toPython())(g.value(s, p))

user_rows = [{'uri': str(s), 'id': val(s, RBO.mbid), 'name': val(s, RBO.displayName)}
             for s in g.subjects(RDF.type, RES.User)]
album_rows = [{'uri': str(s), 'id': val(s, RBO.mbid), 'name': val(s, RDFS.label)}
              for s in g.subjects(RDF.type, RES.Album)]
review_rows = [{
    'uri': str(s), 'id': val(s, RBO.mbid), 'text': val(s, RBO.reviewText), 'rating': val(s, RBO.rating),
    'language': val(s, RBO.language), 'license': val(s, RBO.license), 'source': val(s, RBO.reviewSource),
    'published_on': str(g.value(s, RBO.publishedOn)) if g.value(s, RBO.publishedOn) else None,
    'url': str(g.value(s, RBO.reviewUrl)), 'match_method': val(s, RBO.matchMethod),
} for s in g.subjects(RDF.type, RES.Review)]

mbids_por_track = defaultdict(list)
for s, o in g.subject_objects(RBO.recordingMbid):
    mbids_por_track[str(s)].append(str(o))
recording_rows = [{'uri': u, 'mbids': sorted(v)} for u, v in mbids_por_track.items()]

def filas_rel(pred):
    return [{'s': str(s), 'o': str(o)} for s, o in g.subject_objects(pred)]

RELS = [   # (predicado, etiqueta origen, etiqueta destino)
    (RBO.writtenBy, 'Review', 'User'), (RBO.reviewsRecording, 'Review', 'Track'),
    (RBO.reviewsAlbum, 'Review', 'Album'), (RBO.reviewsArtist, 'Review', 'Artist'),
    (RBO.appearsOn, 'Track', 'Album'), (RBO.releasedBy, 'Album', 'Artist'),
]
print(f'Usuarios {len(user_rows):,} | Reseñas {len(review_rows):,} | Álbumes {len(album_rows):,} | '
      f'Canciones con recording MBID {len(recording_rows):,}')

Usuarios 387 | Reseñas 2,006 | Álbumes 1,849 | Canciones con recording MBID 24,910


In [29]:
for etiqueta in ('User', 'Review', 'Album'):
    run_cypher(f'CREATE CONSTRAINT {etiqueta.lower()}_uri_unique IF NOT EXISTS FOR (n:{etiqueta}) REQUIRE n.uri IS UNIQUE')
run_cypher('CREATE CONSTRAINT chunk_id_unique IF NOT EXISTS FOR (n:Chunk) REQUIRE n.id IS UNIQUE')

cargar_en_lotes('Nodos :User', 'UNWIND $rows AS row MERGE (n:User {uri: row.uri}) SET n += row', user_rows)
cargar_en_lotes('Nodos :Album', 'UNWIND $rows AS row MERGE (n:Album {uri: row.uri}) SET n += row', album_rows)
cargar_en_lotes('Nodos :Review', 'UNWIND $rows AS row MERGE (n:Review {uri: row.uri}) SET n += row', review_rows, 200)
cargar_en_lotes('Track.recordingMbids', '''UNWIND $rows AS row MATCH (t:Track {uri: row.uri})
    SET t.recordingMbids = row.mbids''', recording_rows)
for pred, origen, destino in RELS:
    rel = tipo_relacion(local(pred))
    cargar_en_lotes(f'Relaciones :{rel}', f'''UNWIND $rows AS row
        MATCH (a:{origen} {{uri: row.s}}) MATCH (b:{destino} {{uri: row.o}})
        MERGE (a)-[:{rel}]->(b)''', filas_rel(pred))

✓ Nodos :User                                   387  (0.9s)
✓ Nodos :Album                                1,849  (1.7s)
✓ Nodos :Review                               2,006  (15.6s)
✓ Track.recordingMbids                       24,910  (15.0s)
✓ Relaciones :WRITTEN_BY                      2,006  (1.7s)
✓ Relaciones :REVIEWS_RECORDING                  14  (0.6s)
✓ Relaciones :REVIEWS_ALBUM                   1,942  (1.2s)
✓ Relaciones :REVIEWS_ARTIST                     50  (0.6s)
✓ Relaciones :APPEARS_ON                      2,981  (1.8s)
✓ Relaciones :RELEASED_BY                     1,859  (1.2s)


### 5.1 Materialización de la preferencia `INTERACTED_WITH`

Se aplica en Cypher cada cadena de propiedades de `rbo:interactedWith` (razonamiento "hacia adelante"). La relación guarda
por qué existe (`via`), la calificación del usuario (si la dio) y la reseña de origen, para poder explicar recomendaciones.

In [30]:
CADENAS = [
    ('recording', 'MATCH (u:User)<-[:WRITTEN_BY]-(r:Review)-[:REVIEWS_RECORDING]->(t:Track)'),
    ('album',     'MATCH (u:User)<-[:WRITTEN_BY]-(r:Review)-[:REVIEWS_ALBUM]->(:Album)<-[:APPEARS_ON]-(t:Track)'),
    ('artist',    'MATCH (u:User)<-[:WRITTEN_BY]-(r:Review)-[:REVIEWS_ARTIST]->(:Artist)<-[:SUNG_BY]-(t:Track)'),
]
for via, patron in CADENAS:
    n = run_cypher(patron + '''
        MERGE (u)-[i:INTERACTED_WITH {review: r.id}]->(t)
        SET i.via = $via, i.rating = r.rating, i.has_text = r.text IS NOT NULL
        RETURN count(i) AS n''', via=via)[0]['n']
    print(f'✓ INTERACTED_WITH vía {via:<10} {n:>6,}')

✓ INTERACTED_WITH vía recording      14
✓ INTERACTED_WITH vía album       3,131
✓ INTERACTED_WITH vía artist        325


In [31]:
# Meta-grafo: se añaden a la ontología cargada en el punto 1 las clases/propiedades de la extensión
ext_clases = [{'uri': str(c), 'name': local(c), 'declared': True,
               'label_es': next((str(o) for o in onto_ext.objects(c, RDFS.label) if o.language == 'es'), None),
               'comment_es': next((str(o) for o in onto_ext.objects(c, RDFS.comment)), None)}
              for c in onto_ext.subjects(RDF.type, RDFS.Class)]
props = set(onto_ext.subjects(RDF.type, OWL.ObjectProperty)) | set(onto_ext.subjects(RDF.type, OWL.DatatypeProperty))
MAPEO = {p: f'(:{o})-[:{tipo_relacion(local(p))}]->(:{d})' for p, o, d in RELS}
MAPEO[RBO.interactedWith] = '(:User)-[:INTERACTED_WITH]->(:Track)'
MAPEO[RBO.recordingMbid] = 'Track.recordingMbids'
ext_props = [{'uri': str(p), 'name': local(p), 'declared': True,
              'kind': 'ObjectProperty' if (p, RDF.type, OWL.ObjectProperty) in onto_ext else 'DatatypeProperty',
              'neo4j_mapping': MAPEO.get(p)} for p in props]
par = lambda pred: [{'s': str(s), 'o': str(o)} for s, o in onto_ext.subject_objects(pred) if isinstance(o, URIRef)]
cadenas = [{'s': str(p), 'o': str(x), 'chain': i + 1, 'order': j + 1}
           for i, lista in enumerate(onto_ext.objects(RBO.interactedWith, OWL.propertyChainAxiom))
           for j, x in enumerate(Collection(onto_ext, lista))]

PASOS = [
    ('Ontología: clases', 'UNWIND $rows AS row MERGE (n:OntologyClass {uri: row.uri}) SET n += row', ext_clases),
    ('Ontología: propiedades', 'UNWIND $rows AS row MERGE (n:OntologyProperty {uri: row.uri}) SET n += row', ext_props),
    ('Ontología: propiedades inversas', '''UNWIND $rows AS row MERGE (a:OntologyProperty {uri: row.s})
        ON CREATE SET a.name = split(row.s, '#')[1], a.declared = false, a.kind = 'Property'
        WITH a, row MATCH (b:OntologyProperty {uri: row.o}) MERGE (a)-[:INVERSE_OF]->(b)''', par(OWL.inverseOf)),
    ('Ontología: DOMAIN', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:OntologyClass {uri: row.o})
        MERGE (a)-[:DOMAIN]->(b)''', [r for r in par(RDFS.domain)]),
    ('Ontología: RANGE (clases)', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:OntologyClass {uri: row.o})
        MERGE (a)-[:RANGE]->(b)''', [r for r in par(RDFS.range) if not r['o'].startswith(str(XSD))]),
    ('Ontología: RANGE (tipos de dato)', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s})
        MERGE (b:Datatype {uri: row.o}) ON CREATE SET b.name = 'xsd:' + split(row.o, '#')[1]
        MERGE (a)-[:RANGE]->(b)''', [r for r in par(RDFS.range) if r['o'].startswith(str(XSD))]),
    ('Ontología: SUBPROPERTY_OF', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:OntologyProperty {uri: row.o})
        MERGE (a)-[:SUBPROPERTY_OF]->(b)''', par(RDFS.subPropertyOf)),
    ('Ontología: DISJOINT_WITH', '''UNWIND $rows AS row MATCH (a:OntologyClass {uri: row.s}), (b:OntologyClass {uri: row.o})
        MERGE (a)-[:DISJOINT_WITH]->(b)''', par(OWL.disjointWith)),
    ('Ontología: CHAIN_STEP', '''UNWIND $rows AS row MATCH (a:OntologyProperty {uri: row.s}), (b:OntologyProperty {uri: row.o})
        MERGE (a)-[:CHAIN_STEP {chain: row.chain, order: row.order}]->(b)''', cadenas),
]
for nombre, query, rows in PASOS:
    cargar_en_lotes(nombre, query, rows)

✓ Ontología: clases                               3  (0.5s)
✓ Ontología: propiedades                         21  (0.6s)
✓ Ontología: propiedades inversas                 2  (0.8s)
✓ Ontología: DOMAIN                              18  (0.6s)
✓ Ontología: RANGE (clases)                       7  (0.6s)
✓ Ontología: RANGE (tipos de dato)               11  (0.6s)
✓ Ontología: SUBPROPERTY_OF                       3  (0.6s)
✓ Ontología: DISJOINT_WITH                        9  (0.5s)
✓ Ontología: CHAIN_STEP                           8  (0.6s)


### 5.2 Verificación de la carga

In [32]:
esperado = {
    'User': len(user_rows), 'Review': len(review_rows), 'Album': len(album_rows),
    'WRITTEN_BY': len(filas_rel(RBO.writtenBy)), 'REVIEWS_RECORDING': len(filas_rel(RBO.reviewsRecording)),
    'REVIEWS_ALBUM': len(filas_rel(RBO.reviewsAlbum)), 'REVIEWS_ARTIST': len(filas_rel(RBO.reviewsArtist)),
    'APPEARS_ON': len(filas_rel(RBO.appearsOn)), 'RELEASED_BY': len(filas_rel(RBO.releasedBy)),
}
filas = []
for nombre, n_esp in esperado.items():
    q = (f'MATCH (n:{nombre}) RETURN count(n) AS n' if nombre[0].isupper() and not nombre.isupper()
         else f'MATCH ()-[r:{nombre}]->() RETURN count(r) AS n')
    n = run_cypher(q)[0]['n']
    filas.append({'elemento': nombre, 'esperado_rdf': n_esp, 'neo4j': n, 'ok': n == n_esp})
n = run_cypher('MATCH (t:Track) WHERE t.recordingMbids IS NOT NULL RETURN count(t) AS n')[0]['n']
filas.append({'elemento': 'Track.recordingMbids', 'esperado_rdf': len(recording_rows), 'neo4j': n, 'ok': n == len(recording_rows)})
verif = pd.DataFrame(filas)
display(verif)
assert verif.ok.all(), 'Los conteos no coinciden. Revisa la carga.'
print('✓ Todos los conteos coinciden con el RDF.')

,elemento,esperado_rdf,neo4j,ok
0,User,387,387,True
1,Review,2006,2006,True
2,Album,1849,1849,True
3,WRITTEN_BY,2006,2006,True
4,REVIEWS_RECORDING,14,14,True
5,REVIEWS_ALBUM,1942,1942,True
6,REVIEWS_ARTIST,50,50,True
7,APPEARS_ON,2981,2981,True
8,RELEASED_BY,1859,1859,True
9,Track.recordingMbids,24910,24910,True


✓ Todos los conteos coinciden con el RDF.


In [33]:
print('Usuarios más activos y alcance de sus preferencias')
display(pd.DataFrame(run_cypher('''
MATCH (u:User)<-[:WRITTEN_BY]-(r:Review)
OPTIONAL MATCH (u)-[i:INTERACTED_WITH]->(t:Track)
RETURN u.name AS usuario, count(DISTINCT r) AS reseñas, count(DISTINCT t) AS canciones_alcanzadas,
       round(avg(r.rating), 2) AS calificación_media
ORDER BY reseñas DESC LIMIT 10''')))

print('Ejemplo: una canción, su álbum, la reseña y el usuario')
display(pd.DataFrame(run_cypher('''
MATCH (u:User)<-[:WRITTEN_BY]-(r:Review)-[:REVIEWS_ALBUM]->(al:Album)<-[:APPEARS_ON]-(t:Track)-[:SUNG_BY]->(a:Artist)
WHERE r.text IS NOT NULL
RETURN t.title AS canción, a.name AS artista, al.name AS álbum, u.name AS usuario, r.rating AS rating,
       left(r.text, 160) + '…' AS reseña
ORDER BY t.id LIMIT 5''')))

Usuarios más activos y alcance de sus preferencias


,usuario,reseñas,canciones_alcanzadas,calificación_media
0,Chris Jones,177,284,NaN
1,Mike Diver,83,150,NaN
2,Daryl Easlea,73,104,NaN
3,Jack Smith,47,74,NaN
4,Jon Lusk,41,62,NaN
5,Lou Thomas,41,72,NaN
6,smcamp1234,39,55,4.23
7,Ian Wade,37,63,NaN
8,Sid Smith,37,60,NaN
9,sound.and.vision,35,55,2.89


Ejemplo: una canción, su álbum, la reseña y el usuario


,canción,artista,álbum,usuario,rating,reseña
0,She Fell Away (2009 Digital Remaster),Nick Cave & The Bad Seeds,Your Funeral… My Trial,Chris Long,None,Was there ever a bleaker title than Your Funeral... My Trial? Even by Nick Cave's dark-hearted s...
1,What's Happening Brother,Marvin Gaye,What’s Going On,David Katz,None,"Easily one of the greatest albums of all time, What's Going On is nothing short of a masterpiece..."
2,What's Happening Brother,Marvin Gaye,What’s Going On,David O'Donnell,None,"At a time when Motown was better known for pop hits than music with a message, Marvin Gaye's Wha..."
3,No Aloha,The Breeders,Last Splash,Stevie Chick,None,"It's a little known fact that The Breeders' second album - a curate's egg of twisted pop, weird ..."
4,Can't Stop featuring Angela Johnson,Reel People,Second Guess,Jack Smith,None,"Reel People are officially comprised of Papa Records boss Oli Lazarus, engineer Tom Davidson and..."


## 6. Corpus para RAG

**Estrategia de preparación**

1. **Documentos:** reseñas emparejadas **con texto de al menos 10 palabras** (las que solo tienen calificación, o un
   texto trivial como *"Great!"*, quedan como preferencia en el grafo pero no aportan al corpus).
2. **Limpieza:** los enlaces markdown `[texto](url)` se reducen a `texto`; se eliminan URLs sueltas, marcas `*`/`_`/`#` y espacios repetidos.
3. **Segmentación (chunking) por oraciones:** se agrupan oraciones hasta ~120 palabras, con **solapamiento de una oración**
   entre chunks consecutivos para no cortar ideas; un resto de menos de 40 palabras se une al chunk anterior.
   Las reseñas (mediana ≈ 340 palabras) quedan en 2–4 chunks: lo bastante cortos para ser específicos y lo bastante
   largos para conservar contexto.
4. **Contexto en cada chunk:** se antepone un encabezado con la entidad reseñada (p. ej. *"Reseña del álbum «X» de Y"*)
   al texto que se vectoriza, para que fragmentos que no nombran al artista sigan siendo recuperables por él.
5. **Embeddings:** `intfloat/multilingual-e5-small` (384 dimensiones, local y gratuito). Es **multilingüe**: permite
   consultas en español sobre reseñas mayoritariamente en inglés. Usa los prefijos `passage:` / `query:` del modelo.
6. **Almacenamiento:** `(:Chunk {text, context, order, embedding})-[:PART_OF]->(:Review)` en Neo4j, con un **índice vectorial**
   (coseno). Así la recuperación puede combinar similitud semántica con el grafo (filtrar por canciones candidatas).

In [34]:
def limpiar(texto):
    t = re.sub(r'\[([^\]]+)\]\((?:[^()]|\([^)]*\))*\)', r'\1', texto)    # [texto](url) -> texto
    t = re.sub(r'https?://\S+', ' ', t)
    t = re.sub(r'[*_#>]+', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def segmentar(texto, objetivo=120, minimo=40):
    oraciones = []
    for o in re.split(r'(?<=[.!?…])\s+(?=[\"“‘(\[A-ZÁÉÍÓÚÑ0-9¿¡])', texto):
        palabras_o = o.split()
        # Una "oración" demasiado larga (texto sin puntuación) se corta en ventanas de `objetivo` palabras
        oraciones += [' '.join(palabras_o[i:i + objetivo]) for i in range(0, len(palabras_o), objetivo)]
    palabras = lambda xs: sum(len(x.split()) for x in xs)
    chunks, actual, nuevas = [], [], 0           # nuevas = oraciones aún no emitidas en ningún chunk
    for o in oraciones:
        actual.append(o)
        nuevas += 1
        if palabras(actual) >= objetivo:
            chunks.append(' '.join(actual))
            actual = [o] if len(o.split()) < objetivo / 2 else []   # solapamiento: se repite la última oración
            nuevas = 0
    if nuevas:
        pendientes = actual[-nuevas:]
        if chunks and palabras(pendientes) < minimo:
            chunks[-1] += ' ' + ' '.join(pendientes)                 # resto corto: se une al chunk anterior
        else:
            chunks.append(' '.join(actual))
    return chunks or [texto]

In [35]:
nombre_artista = tracks.drop_duplicates('artist_mbid').set_index('artist_mbid').artist.to_dict()
titulo_track = tracks.set_index('track_id')
def contexto(r):
    if r.entity_type == 'release_group':
        meta = MB_META['release_group'][r.entity_id]
        artistas = ', '.join(nombre_artista[a] for a in meta['artists'] if a in nombre_artista)
        return f'Reseña del álbum «{meta["title"]}» de {artistas}'
    if r.entity_type == 'artist':
        return f'Reseña del artista {nombre_artista[r.entity_id]}'
    t = titulo_track.loc[r.track_ids[0]]
    return f'Reseña de la canción «{t.title}» de {t.artist}'

MIN_PALABRAS = 10
documentos = con_texto[con_texto.text.map(lambda t: len(limpiar(t).split()) >= MIN_PALABRAS)]
corpus = []
for r in documentos.itertuples():
    ctx = contexto(r)
    for i, trozo in enumerate(segmentar(limpiar(r.text))):
        corpus.append({'id': f'{r.id}_{i}', 'review_id': r.id, 'review_uri': str(RES[f'Review_{r.id}']),
                       'order': i, 'context': ctx, 'text': trozo, 'n_words': len(trozo.split()),
                       'language': r.language, 'entity_type': r.entity_type})
corpus_df = pd.DataFrame(corpus)
with gzip.open(ARCHIVO_CORPUS, 'wt', encoding='utf-8') as f:
    for c in corpus:
        f.write(json.dumps(c, ensure_ascii=False) + '\n')

print(f'Documentos: {documentos.shape[0]:,} reseñas (de {len(con_texto):,} con texto) -> {len(corpus_df):,} chunks (guardados en {ARCHIVO_CORPUS})')
display(corpus_df.n_words.describe().round(1).to_frame('palabras por chunk').T)
display(corpus_df.groupby('review_id').size().value_counts().sort_index().rename('reseñas').rename_axis('chunks por reseña').to_frame().T)
ej = corpus_df[corpus_df.review_id == corpus_df.review_id.iloc[0]]
for c in ej.itertuples():
    print(f'\n[{c.id}] {c.context}\n{c.text[:400]}…')

Documentos: 1,855 reseñas (de 1,867 con texto) -> 5,335 chunks (guardados en ../data/corpus_rag.jsonl.gz)


,count,mean,std,min,25%,50%,75%,max
palabras por chunk,5335.0,128.9,24.7,10.0,122.0,131.0,142.0,227.0


chunks por reseña,1,2,3,4,5,6,8,9
reseñas,226,232,1000,359,29,5,2,2



[edeecc8a-8149-47ec-9bf6-70ef67752dc1_0] Reseña de la canción «Love Ain't No Stranger (2003 Digital Remaster)» de Whitesnake
I can hear the passion, heart and soul when David sings. The drums are amazing on this song too. But most of all, the music video is cringey AF!…


In [36]:
from sentence_transformers import SentenceTransformer

MODELO_EMBEDDINGS = 'intfloat/multilingual-e5-small'
modelo = SentenceTransformer(MODELO_EMBEDDINGS)
DIM = (modelo.get_embedding_dimension if hasattr(modelo, 'get_embedding_dimension') else modelo.get_sentence_embedding_dimension)()

t0 = time.time()
emb = modelo.encode([f'passage: {c["context"]}. {c["text"]}' for c in corpus],
                    batch_size=64, normalize_embeddings=True, show_progress_bar=True)
print(f'Embeddings: {emb.shape} en {time.time() - t0:.0f}s')

/Users/gabrielgomezcorredor/Desktop/Gabriel_Gomez/Maestria MISIS/Semantic Web/semantic_web_project/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1124 16:04:52.242000 89814 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Batches: 100%|██████████| 84/84 [00:17<00:00,  4.87it/s]

Embeddings: (5335, 384) en 17s


In [37]:
# Re-ejecuciones: se eliminan chunks de una versión anterior del corpus que ya no existen
run_cypher('MATCH (c:Chunk) WHERE NOT c.id IN $ids DETACH DELETE c', ids=[c['id'] for c in corpus])
chunk_rows = [{**{k: c[k] for k in ('id', 'review_uri', 'order', 'context', 'text', 'n_words')},
               'embedding': emb[i].tolist()} for i, c in enumerate(corpus)]
cargar_en_lotes('Nodos :Chunk + PART_OF', '''UNWIND $rows AS row
    MATCH (r:Review {uri: row.review_uri})
    MERGE (c:Chunk {id: row.id})
    SET c.order = row.order, c.context = row.context, c.text = row.text, c.n_words = row.n_words
    WITH c, r, row
    CALL db.create.setNodeVectorProperty(c, 'embedding', row.embedding)
    MERGE (c)-[:PART_OF]->(r)''', chunk_rows, batch_size=200)

run_cypher(f'''CREATE VECTOR INDEX chunk_embedding IF NOT EXISTS FOR (c:Chunk) ON c.embedding
OPTIONS {{indexConfig: {{`vector.dimensions`: {DIM}, `vector.similarity_function`: 'cosine'}}}}''')
run_cypher('CALL db.awaitIndexes(300)')
display(pd.DataFrame(run_cypher('''SHOW VECTOR INDEXES YIELD name, state, populationPercent, labelsOrTypes, properties''')))
n = run_cypher('MATCH (c:Chunk)-[:PART_OF]->(:Review) RETURN count(c) AS n')[0]['n']
assert n == len(corpus), f'Chunks en Neo4j ({n}) != corpus ({len(corpus)})'
print(f'✓ {n:,} chunks con embedding en Neo4j')

✓ Nodos :Chunk + PART_OF                      5,335  (38.3s)


,name,state,populationPercent,labelsOrTypes,properties
0,chunk_embedding,ONLINE,100.0,[Chunk],[embedding]


✓ 5,335 chunks con embedding en Neo4j


## 7. Recuperación de fragmentos relevantes

Dos modos, pensados para el sistema de recomendación con RAG:

- **Global** — `recuperar(consulta)`: búsqueda en el índice vectorial; cada fragmento se devuelve con la entidad reseñada
  y las canciones del grafo a las que se refiere.
- **Restringida a candidatas** — `recuperar_para_canciones(consulta, track_ids)`: dado un conjunto de canciones candidatas
  (p. ej. las que produzca el recomendador), recorre el grafo `Track → (Album | Artist) ← Review ← Chunk` y ordena
  sus fragmentos por similitud con la consulta. Es la pieza que alimentará el contexto del LLM.

In [38]:
def vector_consulta(consulta):
    return modelo.encode(f'query: {consulta}', normalize_embeddings=True).tolist()

def recuperar(consulta, k=5):
    return pd.DataFrame(run_cypher('''
    CALL db.index.vector.queryNodes('chunk_embedding', $k, $qv) YIELD node AS c, score
    MATCH (c)-[:PART_OF]->(r:Review)-[:WRITTEN_BY]->(u:User)
    OPTIONAL MATCH (r)-[:REVIEWS_ALBUM|REVIEWS_ARTIST|REVIEWS_RECORDING]->(e)
    OPTIONAL MATCH (e)<-[:APPEARS_ON|SUNG_BY]-(t:Track)
    WITH c, score, r, u, e, collect(DISTINCT t.title)[0..4] AS canciones
    RETURN round(score, 4) AS score, c.context AS contexto, canciones, u.name AS usuario, r.rating AS rating,
           left(c.text, 220) + '…' AS fragmento
    ORDER BY score DESC''', k=k, qv=vector_consulta(consulta)))

def recuperar_para_canciones(consulta, track_ids, k=5):
    return pd.DataFrame(run_cypher('''
    MATCH (t:Track) WHERE t.id IN $ids
    CALL (t) {
        MATCH (r:Review)-[:REVIEWS_RECORDING]->(t) RETURN r, 'canción' AS nivel
        UNION
        MATCH (r:Review)-[:REVIEWS_ALBUM]->(:Album)<-[:APPEARS_ON]-(t) RETURN r, 'álbum' AS nivel
        UNION
        MATCH (r:Review)-[:REVIEWS_ARTIST]->(:Artist)<-[:SUNG_BY]-(t) RETURN r, 'artista' AS nivel
    }
    MATCH (c:Chunk)-[:PART_OF]->(r)
    WITH t, nivel, c, vector.similarity.cosine(c.embedding, $qv) AS score
    ORDER BY score DESC LIMIT $k
    MATCH (t)-[:SUNG_BY]->(a:Artist)
    RETURN t.id AS track_id, t.title AS canción, a.name AS artista, nivel, round(score, 4) AS score,
           left(c.text, 220) + '…' AS fragmento''', ids=list(track_ids), k=k, qv=vector_consulta(consulta)))

In [39]:
for consulta in ['melancholic album with atmospheric synthesizers and dreamy vocals',
                 'álbum de rock con guitarras potentes y letras políticas',
                 'jazz instrumental relajado con saxofón']:
    print(f'\nConsulta: {consulta}')
    display(recuperar(consulta, k=4))


Consulta: melancholic album with atmospheric synthesizers and dreamy vocals


,score,contexto,canciones,usuario,rating,fragmento
0,0.9159,Reseña del álbum «Quaristice» de Nena,[Simmm],Colin Buttimer,None,"You'll also find brilliant alien landscapes, unexpected dramas, hyperspeed acceleration and much..."
1,0.9147,Reseña del álbum «Forth» de The Verve,[Noise Epic],Chris Jones,None,"His multi-layered, jitttery psychedelia always provides enough distraction to keep the material ..."
2,0.9141,Reseña del álbum «Beast Rest Forth Mouth» de Bear In Heaven,[Beast In Peace],Mike Diver,None,"And another. Then a transfer to the iPod (only 4GB now, so it's keeping some fine company). And ..."
3,0.9117,Reseña del álbum «Memoirs of an Imperfect Angel» de Mariah Carey,[The Impossible],Mike Diver,None,"That's a terrible generalisation, but then again these are terribly middle-of-the-road songs tha..."



Consulta: álbum de rock con guitarras potentes y letras políticas


,score,contexto,canciones,usuario,rating,fragmento
0,0.9334,Reseña del álbum «The Wreckage» de Will Hoge,[Even If It Breaks Your Heart],Sid Smith,None,"Broken romances, bourbon-soaked nights, fighting to stay ahead of the game, overcoming the odds ..."
1,0.9317,Reseña del álbum «The Incident» de Porcupine Tree,"[Drawing the Line (Album Version), Great Expectations (Album Version), Your Unpleasant Family (A...",Raziq Rauf,None,It intersperses jangled acoustic guitars with angular complexities that might fly over some head...
2,0.9316,Reseña del álbum «Let There Be Rockgrass» de Hayseed Dixie,[Dirty Deeds Done Dirt Cheap],Richard Banks,None,"While Barley takes care of guitar and fiddle duties, the Reno brothers provide hillbilly harmoni..."
3,0.9311,Reseña del álbum «Live at the Olympia» de R.E.M.,"[West Of The Fields [Live At The Olympia], Carnival Of Sorts [Box Cars] [Live At The Olympia], W...",Mike Diver,None,"In short, this is a celebration of the career of one of the greatest rock groups America has eve..."



Consulta: jazz instrumental relajado con saxofón


,score,contexto,canciones,usuario,rating,fragmento
0,0.9298,Reseña del álbum «Outer Space / Inner Space» de Flanger,"[The Men Who Fell From Earth, Outer Space/Inner Space]",Peter Marsh,None,"It's jazz, Jim, but not as we know it....Flanger's two previous records on Ntone recast early 19..."
1,0.9296,Reseña del álbum «Sensible Shoes» de Led Bib,[Call Centre Labyrinth (Album Version)],John Eyles,None,The opening track Yes Again has a gentle keyboard intro that builds atmosphere until the whole b...
2,0.9294,Reseña del álbum «Mantis» de Erik Truffaz,[Mare Mosso],Colin Buttimer,None,"So too with this album; ""The Point"" kicks in with said breakbeat for a few bars, then the (doubl..."
3,0.9284,Reseña del álbum «Nuyorican Soul» de Nuyorican Soul,"[Habriendo El Dominate, Maw Latin Blues, Shoshana]",Daryl Easlea,None,"Taking the Salsoul Orchestra's Runaway - first sung by Loleatta Holloway, but here by India - ag..."


In [40]:
# Canciones candidatas de ejemplo: 6 canciones con reseña, de artistas distintos
candidatas = (tracks[tracks.track_id.isin(tracks_de(documentos))].drop_duplicates('artist_mbid')
              .sample(6, random_state=3).track_id.tolist())
print('Candidatas:', candidatas)
display(recuperar_para_canciones('canción alegre y bailable con buena energía', candidatas, k=5))

Candidatas: ['TRACTEB128F4264605', 'TRLGOUL128F1458435', 'TRQLCYH128F92C3EB4', 'TRFYSXX128F42327FD', 'TRFNNTE128F930058E', 'TRGUQFR12903CDAB73']


,track_id,canción,artista,nivel,score,fragmento
0,TRQLCYH128F92C3EB4,Midnight's Another Day,Brian Wilson,álbum,0.9172,"How else could he perform something as trite as Mexican Girl (""Girl, you cast a net/On the day w..."
1,TRQLCYH128F92C3EB4,Midnight's Another Day,Brian Wilson,álbum,0.9118,"All of which begs the question; do we really need more of the airbrushed, idealised California, ..."
2,TRFYSXX128F42327FD,Van Dyke Parks (Album Version),Van Dyke Parks,álbum,0.9116,"It flits impishly between snippets of winsome folk, flourishes of woodwind and clanging percussi..."
3,TRFNNTE128F930058E,Chinatown,Wild Nothing,álbum,0.9112,"The sweetness of Tatum's exquisite melodies, often wrapped tight in a comforting blanket of dist..."
4,TRFYSXX128F42327FD,Van Dyke Parks (Album Version),Van Dyke Parks,álbum,0.9111,"The man who co-wrote Brian Wilson's lost (and rediscovered) opus, The Beach Boys' Smile, must ha..."


### 7.1 Evaluación rápida de la recuperación

Prueba de **auto-recuperación por entidad**: para una muestra de reseñas se consulta *"<álbum/artista> <artista>"* y se
mide si algún fragmento de esa reseña aparece entre los *k* primeros (Recall@k) y en qué posición (MRR).
No mide la calidad final de las recomendaciones (eso se evaluará con el RAG completo), pero valida que el índice,
el contexto de los chunks y los embeddings están funcionando.

In [41]:
random.seed(11)
muestra = random.sample(list(documentos.itertuples()), min(150, len(documentos)))
K = 10
aciertos, rr = Counter(), []
for r in muestra:
    consulta = contexto(r).replace('Reseña del álbum ', '').replace('Reseña del artista ', '').replace('Reseña de la canción ', '')
    res = run_cypher('''CALL db.index.vector.queryNodes('chunk_embedding', $k, $qv) YIELD node AS c, score
                        MATCH (c)-[:PART_OF]->(rv:Review) RETURN rv.id AS review''', k=K, qv=vector_consulta(consulta))
    pos = next((i + 1 for i, x in enumerate(res) if x['review'] == r.id), None)
    rr.append(1 / pos if pos else 0)
    for k in (1, 5, 10):
        aciertos[k] += bool(pos and pos <= k)
display(pd.DataFrame([{**{f'Recall@{k}': round(aciertos[k] / len(muestra), 3) for k in (1, 5, 10)},
                       'MRR': round(sum(rr) / len(rr), 3), 'consultas': len(muestra)}]))

,Recall@1,Recall@5,Recall@10,MRR,consultas
0,0.88,0.913,0.913,0.895,150


## 8. Resumen

In [42]:
resumen = run_cypher('''
CALL () { MATCH (n:User) RETURN count(n) AS usuarios }
CALL () { MATCH (n:Review) RETURN count(n) AS reseñas }
CALL () { MATCH (n:Album) RETURN count(n) AS álbumes }
CALL () { MATCH (n:Chunk) RETURN count(n) AS chunks }
CALL () { MATCH ()-[i:INTERACTED_WITH]->() RETURN count(i) AS interacciones }
CALL () { MATCH (t:Track) WHERE ()-[:INTERACTED_WITH]->(t) RETURN count(t) AS canciones_con_preferencias }
CALL () { MATCH (n) RETURN count(n) AS nodos_totales }
CALL () { MATCH ()-[r]->() RETURN count(r) AS relaciones_totales }
RETURN *''')[0]
display(pd.DataFrame([resumen]).T.rename(columns={0: 'valor'}))
display(cobertura)

,valor
canciones_con_preferencias,3206
chunks,5335
interacciones,3470
nodos_totales,137446
relaciones_totales,147683
reseñas,2006
usuarios,387
álbumes,1849


,estrategia,reseñas,reseñas_con_texto,entidades,usuarios,canciones_cubiertas,canciones_con_texto
0,recording_mbid,7,0,7,3,7,0
1,recording_artista_titulo,7,4,7,5,7,4
2,album_artista_titulo,1942,1845,1849,348,2966,2889
3,artista_mbid,50,18,48,40,313,139
4,TOTAL (sin duplicar),2006,1867,1911,387,3206,2999


**Conclusiones**

- **Fuente integrada:** CritiqueBrainz (reseñas y calificaciones de usuarios, licencias CC BY-SA / BY-NC-SA).
  Los usuarios, sus reseñas y los álbumes reseñados quedaron en Neo4j y en RDF (`reviews.ttl`), descritos por una
  extensión de la ontología (`ontology_reviews.ttl`).
- **Reconciliación:** la vía directa por `recording_mbid` (mapeo MSD → MusicBrainz) es exacta pero casi no aporta:
  CritiqueBrainz tiene muy pocas reseñas de *recordings* y las que coinciden no tienen texto. La cobertura útil viene de
  las reseñas de **álbumes**, cruzadas por **MBID del artista + título normalizado del álbum**; se complementa con
  reseñas de artistas (MBID exacto). Cada reseña conserva su `match_method` para trazabilidad.
- **Preferencias:** la relación `INTERACTED_WITH` (usuario → canción) se definió en OWL con cadenas de propiedades
  y se materializó en Neo4j con su justificación (`via`, `rating`, `review`).
- **Corpus RAG:** reseñas limpias → chunks por oraciones (~120 palabras, solapamiento de una oración, encabezado de
  contexto) → embeddings multilingües → índice vectorial en Neo4j, con recuperación global o restringida a canciones candidatas.
- **Cobertura:** baja en proporción al grafo (ver tabla), como anticipa el enunciado, pero suficiente para construir y
  evaluar el componente RAG: miles de canciones y artistas conocidos tienen al menos una crítica asociada.

In [43]:
driver.close()